In [ ]:
import sys, os
import numpy as np
import pandas as pd
from pathlib import Path

# Locate superligaen/ regardless of where Jupyter was launched
_here = Path.cwd()
if not (_here / 'run_model.py').exists():
    for _c in [_here / 'superligaen',
               _here / 'non_penalty_bayes' / 'superligaen',
               _here / 'team_strength' / 'non_penalty_bayes' / 'superligaen']:
        if _c.exists():
            os.chdir(_c)
            break

NOTEBOOK_DIR = Path.cwd().resolve()
NP_BAYES_DIR = str(NOTEBOOK_DIR.parent)
sys.path.insert(0, NP_BAYES_DIR)

from src.data_utils import load_and_process_data
from src.superliga_model import build_and_sample_model

LEAGUE = 'Superligaen'
SEASON = '2025-2026'

REPO_ROOT = str(NOTEBOOK_DIR.parents[4])
DB_PATH   = os.path.join(REPO_ROOT, 'infra', 'data', 'db', 'fotmob.db')

DECAY_RATE   = 0.0018
GOALS_WEIGHT = 0.27
XG_WEIGHT    = 0.55
PSXG_WEIGHT  = 0.17
EPV_WEIGHT   = 0.0
N_SAMPLES    = 2_000
N_TUNE       = 1_000

print(f'CWD: {NOTEBOOK_DIR}')
print(f'DB:  {DB_PATH}')

In [ ]:
df, team_mapping, n_teams = load_and_process_data(
    db_path=DB_PATH, league=LEAGUE, season=SEASON,
    decay_rate=DECAY_RATE,
    goals_weight=GOALS_WEIGHT, xg_weight=XG_WEIGHT,
    psxg_weight=PSXG_WEIGHT, epv_weight=EPV_WEIGHT,
)
print(f'{n_teams} teams, {df["match_id"].nunique()} matches')

_, trace = build_and_sample_model(
    df, n_teams, trace=N_SAMPLES, tune=N_TUNE, team_mapping=team_mapping,
)
print('Done.')

In [ ]:
# ── Strength ratings table ────────────────────────────────────────────────
posterior = trace.posterior
n_teams   = len(team_names)

att  = posterior['att_str'].values.reshape(-1, n_teams)
defn = posterior['def_str'].values.reshape(-1, n_teams)
hadv = posterior['home_adv'].values.reshape(-1)
base = posterior['baseline'].values.reshape(-1)

ratings = pd.DataFrame({
    'team':     team_names,
    'att_mean': att.mean(axis=0).round(3),
    'att_sd':   att.std(axis=0).round(3),
    'att_lo':   np.percentile(att, 5,  axis=0).round(3),
    'att_hi':   np.percentile(att, 95, axis=0).round(3),
    'def_mean': defn.mean(axis=0).round(3),
    'def_sd':   defn.std(axis=0).round(3),
    'def_lo':   np.percentile(defn, 5,  axis=0).round(3),
    'def_hi':   np.percentile(defn, 95, axis=0).round(3),
})

ratings['net'] = (ratings['att_mean'] - ratings['def_mean']).round(3)
ratings = ratings.sort_values('net', ascending=False).reset_index(drop=True)
ratings

In [ ]:
# ── Expected goals vs average opponent ───────────────────────────────────
n_samp   = 5_000
idx      = np.random.choice(len(base), size=n_samp, replace=True)
team_idx = {name: i for i, name in enumerate(team_names)}

rows = []
for team in team_names:
    ti = team_idx[team]
    rows.append({
        'team':        team,
        'xg_home_for': round(np.exp(base[idx] + att[idx, ti] + hadv[idx]).mean(), 3),
        'xg_home_ag':  round(np.exp(base[idx] + defn[idx, ti]).mean(), 3),
        'xg_away_for': round(np.exp(base[idx] + att[idx, ti]).mean(), 3),
        'xg_away_ag':  round(np.exp(base[idx] + defn[idx, ti] + hadv[idx]).mean(), 3),
    })

xg_table = pd.DataFrame(rows)
xg_table['xg_diff'] = (
    xg_table['xg_home_for'] + xg_table['xg_away_for'] -
    xg_table['xg_home_ag']  - xg_table['xg_away_ag']
).round(3)
xg_table.sort_values('xg_diff', ascending=False).reset_index(drop=True)